# GPU benchmark: CUDA Monte Carlo engine vs single-threaded CPU

Runtime → Change runtime type → **T4 GPU** (or any NVIDIA GPU). Then *Run all*.

The notebook builds the engine with CUDA, runs the GoogleTest suite (GPU tests included),
benchmarks 1M / 2M / 4M paths × 252 steps on both backends and saves `bench.json`.
Copy the printed table into `docs/BENCHMARKS.md`.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2
!lscpu | grep 'Model name'

In [ ]:
!git clone -q https://github.com/SriharshaPotta/options-strategy-backtester.git
%cd options-strategy-backtester
!pip install -q "cmake>=3.24" pybind11

In [ ]:
!cmake -S engine -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=native       -DPython_EXECUTABLE=$(which python) 2>&1 | grep -E "CUDA|Python|Error"
!cmake --build build -j 2>&1 | tail -3

In [ ]:
!ctest --test-dir build --output-on-failure | tail -25

## Benchmark
GPU time = simulation kernel + block reduction + result copy (cudaEvent, median of 5 after warm-up). RNG seeding is reported separately. CPU = single thread, same discretisation, `-O3`.

In [ ]:
!./build/mc_bench --paths 1048576,2097152,4194304 --steps 252 --reps 5 --cpu-reps 3 --json bench.json

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

bench = json.load(open("bench.json"))
df = pd.DataFrame(bench["results"])
print("device:", bench["device"])
cols = ["case", "paths", "cpu_ms", "gpu_ms", "gpu_rng_init_ms", "speedup", "cpu_price", "gpu_price", "bs_reference", "prices_agree"]
display(df[cols])

fig, ax = plt.subplots(figsize=(7, 3.5))
for case, g in df.groupby("case"):
    ax.plot(g["paths"] / 1e6, g["speedup"], marker="o", label=case)
ax.set_xlabel("paths (millions), 252 steps"); ax.set_ylabel("speedup vs 1 CPU thread")
ax.set_title(bench["device"]); ax.grid(alpha=.3); ax.legend()
plt.show()

print(df[["case", "paths", "cpu_ms", "gpu_ms", "speedup"]].to_markdown(index=False, floatfmt=".1f"))

## Python bindings on the GPU

In [ ]:
import sys
sys.path.insert(0, "backtester")
from osb import engine

print("CUDA:", engine.has_cuda(), engine.device_name())
legs = [("put", 85, 1), ("put", 92, -1), ("call", 108, -1), ("call", 115, 1)]
for backend in ("cuda", "cpu"):
    r = engine.price(100, 0.04, 0.2, 1.0, legs, n_paths=1 << 20, n_steps=252, backend=backend)
    print(f"{backend:>4}: price={r['price']:.4f} ± {r['std_error']:.4f}  sim={r['elapsed_ms']:.1f} ms")